## Step 1 — Create the new notebook and load the two files

In [1]:
# ============================================================
# STEP 1: LOAD CURATED DATA + MODEL FEATURE MATRIX
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/Users/maryta/Downloads/mdm2_Reinvent_4_final")

CURATED_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "mdm2_pic50_qsar_standardized.csv"
)

FEATURE_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "qsar_features"
    / "mdm2_padel_1d2d_morgan_features.csv"
)

df_curated = pd.read_csv(CURATED_FILE)
df_features = pd.read_csv(FEATURE_FILE)

print("Curated dataset shape:", df_curated.shape)
print("Feature matrix shape:", df_features.shape)

print("\nCurated columns:")
print(df_curated.columns.tolist())

print("\nFirst feature columns:")
print(df_features.columns[:10].tolist())

Curated dataset shape: (4146, 9)
Feature matrix shape: (4048, 3496)

Curated columns:
['molecule_chembl_id', 'canonical_smiles', 'standardized_smiles', 'IC50_nM_median', 'pIC50', 'pIC50_mean', 'pIC50_std', 'measurement_count', 'bioactivity_class']

First feature columns:
['Name', 'padel_molecule_id', 'standardized_smiles', 'pIC50', 'nAcid', 'ALogP', 'ALogp2', 'AMR', 'apol', 'naAromAtom']


## Step 2 — Align activity labels with the feature matrix and remove the intermediate class

In [2]:
# ============================================================
# STEP 2: ALIGN LABELS + KEEP ACTIVE / INACTIVE ONLY
# ============================================================

# Keep only the columns needed from the curated dataset
label_df = df_curated[
    [
        "standardized_smiles",
        "pIC50",
        "bioactivity_class"
    ]
].copy()

# Merge activity labels onto the existing feature matrix
df_binary = df_features.merge(
    label_df,
    on="standardized_smiles",
    how="left",
    suffixes=("", "_label")
)

print("After merge shape:", df_binary.shape)

print("\nMissing activity labels:")
print(df_binary["bioactivity_class"].isna().sum())

# Keep only active and inactive compounds
df_binary = df_binary[
    df_binary["bioactivity_class"].isin(
        ["active", "inactive"]
    )
].copy()

print("\nBinary classification dataset shape:")
print(df_binary.shape)

print("\nClass counts:")
print(df_binary["bioactivity_class"].value_counts())

print("\nClass percentages:")
print(
    (
        df_binary["bioactivity_class"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

After merge shape: (4048, 3498)

Missing activity labels:
0

Binary classification dataset shape:
(3757, 3498)

Class counts:
bioactivity_class
active      3593
inactive     164
Name: count, dtype: int64

Class percentages:
bioactivity_class
active      95.63
inactive     4.37
Name: proportion, dtype: float64


## Step 3 — generate Bemis–Murcko scaffolds for the binary dataset

In [3]:
# ============================================================
# STEP 3: GENERATE BEMIS-MURCKO SCAFFOLDS
# ============================================================

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


def get_murcko_scaffold(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    return Chem.MolToSmiles(
        scaffold,
        canonical=True
    )


df_binary["murcko_scaffold"] = (
    df_binary["standardized_smiles"]
    .apply(get_murcko_scaffold)
)


print("Missing scaffolds:")
print(df_binary["murcko_scaffold"].isna().sum())

print("\nUnique scaffolds:")
print(df_binary["murcko_scaffold"].nunique())


# Check how many scaffolds contain inactive compounds
scaffold_class_summary = (
    df_binary
    .groupby(
        ["murcko_scaffold", "bioactivity_class"]
    )
    .size()
    .unstack(fill_value=0)
)

print("\nScaffolds containing inactive compounds:")

print(
    (
        scaffold_class_summary
        .get("inactive", 0) > 0
    ).sum()
)

Missing scaffolds:
0

Unique scaffolds:
1367

Scaffolds containing inactive compounds:
68


In [4]:
# ============================================================
# STEP 4: SCAFFOLD-AWARE 70/15/15 SPLIT
# WHILE PRESERVING THE RARE INACTIVE CLASS
# ============================================================

import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Reset index for safe positional assignment
df_binary = df_binary.reset_index(drop=True)

# ------------------------------------------------------------
# 1. Summarize every scaffold
# ------------------------------------------------------------

scaffold_info = (
    df_binary
    .groupby("murcko_scaffold")
    .agg(
        n_compounds=("bioactivity_class", "size"),
        n_inactive=(
            "bioactivity_class",
            lambda x: (x == "inactive").sum()
        ),
        n_active=(
            "bioactivity_class",
            lambda x: (x == "active").sum()
        )
    )
    .reset_index()
)

# Randomize before sorting to avoid deterministic ties
scaffold_info = scaffold_info.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# Prioritize scaffolds containing inactive compounds,
# then larger scaffold groups
scaffold_info = scaffold_info.sort_values(
    ["n_inactive", "n_compounds"],
    ascending=[False, False]
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Target sizes
# ------------------------------------------------------------

n_total = len(df_binary)
n_inactive_total = (df_binary["bioactivity_class"] == "inactive").sum()

targets = {
    "train": {
        "n": round(0.70 * n_total),
        "inactive": round(0.70 * n_inactive_total)
    },
    "validation": {
        "n": round(0.15 * n_total),
        "inactive": round(0.15 * n_inactive_total)
    },
    "test": {
        "n": n_total
             - round(0.70 * n_total)
             - round(0.15 * n_total),

        "inactive": n_inactive_total
                    - round(0.70 * n_inactive_total)
                    - round(0.15 * n_inactive_total)
    }
}

print("Targets:")
print(targets)


# ------------------------------------------------------------
# 3. Greedy scaffold assignment
# ------------------------------------------------------------

assigned = {
    "train": [],
    "validation": [],
    "test": []
}

current_n = {
    "train": 0,
    "validation": 0,
    "test": 0
}

current_inactive = {
    "train": 0,
    "validation": 0,
    "test": 0
}


for _, row in scaffold_info.iterrows():

    scores = {}

    for split in ["train", "validation", "test"]:

        projected_n = current_n[split] + row["n_compounds"]
        projected_inactive = (
            current_inactive[split]
            + row["n_inactive"]
        )

        size_error = abs(
            projected_n - targets[split]["n"]
        ) / targets[split]["n"]

        inactive_target = max(
            targets[split]["inactive"],
            1
        )

        inactive_error = abs(
            projected_inactive
            - targets[split]["inactive"]
        ) / inactive_target

        # Give extra importance to minority-class balance
        scores[split] = (
            size_error
            + 2.0 * inactive_error
        )

    best_split = min(
        scores,
        key=scores.get
    )

    assigned[best_split].append(
        row["murcko_scaffold"]
    )

    current_n[best_split] += row["n_compounds"]
    current_inactive[best_split] += row["n_inactive"]


# ------------------------------------------------------------
# 4. Assign labels back to molecules
# ------------------------------------------------------------

scaffold_to_split = {}

for split, scaffolds in assigned.items():

    for scaffold in scaffolds:
        scaffold_to_split[scaffold] = split


df_binary["split"] = (
    df_binary["murcko_scaffold"]
    .map(scaffold_to_split)
)


# ------------------------------------------------------------
# 5. Report split composition
# ------------------------------------------------------------

print("\nSplit sizes:")
print(df_binary["split"].value_counts())

print("\nClass counts by split:")
print(
    pd.crosstab(
        df_binary["split"],
        df_binary["bioactivity_class"]
    )
)

print("\nClass percentages within each split:")
print(
    pd.crosstab(
        df_binary["split"],
        df_binary["bioactivity_class"],
        normalize="index"
    ).mul(100).round(2)
)

Targets:
{'train': {'n': 2630, 'inactive': 115}, 'validation': {'n': 564, 'inactive': 25}, 'test': {'n': 563, 'inactive': np.int64(24)}}

Split sizes:
split
train         3596
validation      90
test            71
Name: count, dtype: int64

Class counts by split:
bioactivity_class  active  inactive
split                              
test                   23        48
train                3529        67
validation             41        49

Class percentages within each split:
bioactivity_class  active  inactive
split                              
test                32.39     67.61
train               98.14      1.86
validation          45.56     54.44


In [5]:
# ============================================================
# STEP 4 — STRATIFIED + SCAFFOLD-AWARE SPLIT
# 20 folds -> 14 train / 3 validation / 3 test
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df_binary = df_binary.reset_index(drop=True)

X_dummy = np.zeros(len(df_binary))

y_split = (
    df_binary["bioactivity_class"]
    .map({
        "inactive": 0,
        "active": 1
    })
    .to_numpy()
)

groups = df_binary["murcko_scaffold"].to_numpy()


# ------------------------------------------------------------
# 1. Create 20 scaffold-aware stratified folds
# ------------------------------------------------------------

sgkf = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=42
)

fold_id = np.full(
    len(df_binary),
    -1,
    dtype=int
)

for fold, (_, fold_indices) in enumerate(
    sgkf.split(
        X_dummy,
        y_split,
        groups=groups
    )
):
    fold_id[fold_indices] = fold


df_binary["fold"] = fold_id


# ------------------------------------------------------------
# 2. Combine folds into ~70/15/15
# ------------------------------------------------------------

# 14/20 = 70%
train_folds = list(range(0, 14))

# 3/20 = 15%
validation_folds = list(range(14, 17))

# 3/20 = 15%
test_folds = list(range(17, 20))


df_binary["split"] = np.select(
    [
        df_binary["fold"].isin(train_folds),
        df_binary["fold"].isin(validation_folds),
        df_binary["fold"].isin(test_folds)
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="unassigned"
)


# ------------------------------------------------------------
# 3. Report results
# ------------------------------------------------------------

print("Split sizes:")
print(
    df_binary["split"]
    .value_counts()
)

print("\nClass counts:")
print(
    pd.crosstab(
        df_binary["split"],
        df_binary["bioactivity_class"]
    )
)

print("\nClass percentages:")
print(
    pd.crosstab(
        df_binary["split"],
        df_binary["bioactivity_class"],
        normalize="index"
    ).mul(100).round(2)
)

Split sizes:
split
train         2619
test           571
validation     567
Name: count, dtype: int64

Class counts:
bioactivity_class  active  inactive
split                              
test                  539        32
train                2509       110
validation            545        22

Class percentages:
bioactivity_class  active  inactive
split                              
test                94.40      5.60
train               95.80      4.20
validation          96.12      3.88


In [6]:
# ============================================================
# STEP 5: CHECK SCAFFOLD LEAKAGE
# ============================================================

train_scaffolds = set(
    df_binary.loc[
        df_binary["split"] == "train",
        "murcko_scaffold"
    ]
)

val_scaffolds = set(
    df_binary.loc[
        df_binary["split"] == "validation",
        "murcko_scaffold"
    ]
)

test_scaffolds = set(
    df_binary.loc[
        df_binary["split"] == "test",
        "murcko_scaffold"
    ]
)

print(
    "Train–Validation overlap:",
    len(train_scaffolds & val_scaffolds)
)

print(
    "Train–Test overlap:",
    len(train_scaffolds & test_scaffolds)
)

print(
    "Validation–Test overlap:",
    len(val_scaffolds & test_scaffolds)
)

Train–Validation overlap: 0
Train–Test overlap: 0
Validation–Test overlap: 0


In [9]:
# ============================================================
# STEP 6: PREPARE X / y FOR BINARY CLASSIFICATION
# ============================================================

# Columns that should NOT be used as model features
non_feature_columns = [
    "Name",
    "padel_molecule_id",
    "standardized_smiles",
    "pIC50",
    "pIC50_label",
    "bioactivity_class",
    "murcko_scaffold",
    "fold",
    "split",
]

feature_columns = [
    col for col in df_binary.columns
    if col not in non_feature_columns
]

print("Number of candidate features:", len(feature_columns))


# ------------------------------------------------------------
# Split datasets
# ------------------------------------------------------------

train_df = df_binary[
    df_binary["split"] == "train"
].copy()

val_df = df_binary[
    df_binary["split"] == "validation"
].copy()

test_df = df_binary[
    df_binary["split"] == "test"
].copy()


# ------------------------------------------------------------
# X matrices
# ------------------------------------------------------------

X_train = train_df[feature_columns].copy()
X_val   = val_df[feature_columns].copy()
X_test  = test_df[feature_columns].copy()


# ------------------------------------------------------------
# Binary target
# inactive = 0
# active   = 1
# ------------------------------------------------------------

label_map = {
    "inactive": 0,
    "active": 1
}

y_train = (
    train_df["bioactivity_class"]
    .map(label_map)
    .astype(int)
)

y_val = (
    val_df["bioactivity_class"]
    .map(label_map)
    .astype(int)
)

y_test = (
    test_df["bioactivity_class"]
    .map(label_map)
    .astype(int)
)


print("\nShapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:", y_val.shape)
print("X_test: ", X_test.shape,  "y_test:", y_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

Number of candidate features: 3492

Shapes:
X_train: (2619, 3492) y_train: (2619,)
X_val:   (567, 3492) y_val: (567,)
X_test:  (571, 3492) y_test: (571,)

Training labels:
bioactivity_class
1    2509
0     110
Name: count, dtype: int64


In [10]:
# ============================================================
# STEP 7: TRAINING-ONLY PREPROCESSING
# ============================================================

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold


# ------------------------------------------------------------
# 1. Remove features with >20% missing values in TRAIN only
# ------------------------------------------------------------

missing_fraction_train = X_train.isna().mean()

keep_mask = missing_fraction_train <= 0.20
kept_columns = X_train.columns[keep_mask]

X_train_f = X_train[kept_columns].copy()
X_val_f   = X_val[kept_columns].copy()
X_test_f  = X_test[kept_columns].copy()

print("Original features:", X_train.shape[1])
print("After >20% missing filter:", X_train_f.shape[1])


# ------------------------------------------------------------
# 2. Median imputation — fit on TRAIN only
# ------------------------------------------------------------

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train_f)
X_val_imp   = imputer.transform(X_val_f)
X_test_imp  = imputer.transform(X_test_f)


# ------------------------------------------------------------
# 3. Remove zero-variance features — fit on TRAIN only
# ------------------------------------------------------------

variance_filter = VarianceThreshold(threshold=0.0)

X_train_proc = variance_filter.fit_transform(X_train_imp)
X_val_proc   = variance_filter.transform(X_val_imp)
X_test_proc  = variance_filter.transform(X_test_imp)


# ------------------------------------------------------------
# 4. Recover final feature names
# ------------------------------------------------------------

final_feature_names = (
    kept_columns[
        variance_filter.get_support()
    ]
    .tolist()
)


# ------------------------------------------------------------
# 5. Checks
# ------------------------------------------------------------

print("After zero-variance removal:", len(final_feature_names))

print("\nFinal matrix shapes:")
print("Train:", X_train_proc.shape)
print("Validation:", X_val_proc.shape)
print("Test:", X_test_proc.shape)

print("\nRemaining NaNs:")
print(
    np.isnan(X_train_proc).sum(),
    np.isnan(X_val_proc).sum(),
    np.isnan(X_test_proc).sum()
)

Original features: 3492
After >20% missing filter: 3393


ValueError: Input X contains infinity or a value too large for dtype('float64').

In [11]:
# ============================================================
# DIAGNOSE IMPUTER ERROR
# ============================================================

import numpy as np
import pandas as pd

print("Train shape:", X_train_f.shape)
print("Validation shape:", X_val_f.shape)
print("Test shape:", X_test_f.shape)

print("\nSame train/validation columns:",
      X_train_f.columns.equals(X_val_f.columns))

print("Same train/test columns:",
      X_train_f.columns.equals(X_test_f.columns))


# Check non-numeric columns
train_non_numeric = X_train_f.select_dtypes(
    exclude=[np.number]
).columns.tolist()

val_non_numeric = X_val_f.select_dtypes(
    exclude=[np.number]
).columns.tolist()

test_non_numeric = X_test_f.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nNon-numeric columns:")
print("Train:", train_non_numeric)
print("Validation:", val_non_numeric)
print("Test:", test_non_numeric)


# Check infinity
for name, X in [
    ("TRAIN", X_train_f),
    ("VALIDATION", X_val_f),
    ("TEST", X_test_f)
]:
    numeric = X.select_dtypes(include=[np.number])

    print(f"\n{name}")
    print("NaN:", numeric.isna().sum().sum())
    print("+inf:", np.isposinf(numeric.to_numpy()).sum())
    print("-inf:", np.isneginf(numeric.to_numpy()).sum())

Train shape: (2619, 3393)
Validation shape: (567, 3393)
Test shape: (571, 3393)

Same train/validation columns: True
Same train/test columns: True

Non-numeric columns:
Train: []
Validation: []
Test: []

TRAIN
NaN: 328662
+inf: 0
-inf: 0

VALIDATION
NaN: 57511
+inf: 228
-inf: 0

TEST
NaN: 61201
+inf: 0
-inf: 0


In [12]:
# ============================================================
# STEP 7 FIX: HANDLE INF + TRAINING-ONLY PREPROCESSING
# ============================================================

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold


# ------------------------------------------------------------
# 1. Replace +/- inf with NaN
# ------------------------------------------------------------

X_train_clean = X_train.replace(
    [np.inf, -np.inf],
    np.nan
).copy()

X_val_clean = X_val.replace(
    [np.inf, -np.inf],
    np.nan
).copy()

X_test_clean = X_test.replace(
    [np.inf, -np.inf],
    np.nan
).copy()


# ------------------------------------------------------------
# 2. Remove features with >20% missing in TRAIN only
# ------------------------------------------------------------

missing_fraction_train = (
    X_train_clean
    .isna()
    .mean()
)

keep_mask = (
    missing_fraction_train <= 0.20
)

kept_columns = (
    X_train_clean
    .columns[keep_mask]
)

X_train_f = X_train_clean[kept_columns].copy()
X_val_f   = X_val_clean[kept_columns].copy()
X_test_f  = X_test_clean[kept_columns].copy()

print(
    "Original features:",
    X_train.shape[1]
)

print(
    "After >20% missing filter:",
    X_train_f.shape[1]
)


# ------------------------------------------------------------
# 3. Median imputation
# FIT ON TRAIN ONLY
# ------------------------------------------------------------

imputer = SimpleImputer(
    strategy="median"
)

X_train_imp = imputer.fit_transform(
    X_train_f
)

X_val_imp = imputer.transform(
    X_val_f
)

X_test_imp = imputer.transform(
    X_test_f
)


# ------------------------------------------------------------
# 4. Remove zero-variance features
# FIT ON TRAIN ONLY
# ------------------------------------------------------------

variance_filter = VarianceThreshold(
    threshold=0.0
)

X_train_proc = variance_filter.fit_transform(
    X_train_imp
)

X_val_proc = variance_filter.transform(
    X_val_imp
)

X_test_proc = variance_filter.transform(
    X_test_imp
)


# ------------------------------------------------------------
# 5. Recover final feature names
# ------------------------------------------------------------

final_feature_names = (
    kept_columns[
        variance_filter.get_support()
    ]
    .tolist()
)


# ------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------

print(
    "After zero-variance removal:",
    len(final_feature_names)
)

print("\nFinal matrix shapes:")
print("Train:", X_train_proc.shape)
print("Validation:", X_val_proc.shape)
print("Test:", X_test_proc.shape)

print("\nRemaining NaNs:")
print(
    np.isnan(X_train_proc).sum(),
    np.isnan(X_val_proc).sum(),
    np.isnan(X_test_proc).sum()
)

print("\nRemaining inf values:")
print(
    np.isinf(X_train_proc).sum(),
    np.isinf(X_val_proc).sum(),
    np.isinf(X_test_proc).sum()
)

Original features: 3492
After >20% missing filter: 3393
After zero-variance removal: 3088

Final matrix shapes:
Train: (2619, 3088)
Validation: (567, 3088)
Test: (571, 3088)

Remaining NaNs:
0 0 0

Remaining inf values:
0 0 0


In [15]:
# ============================================================
# STEP 7B: REMOVE INVALID EXTREME PaDEL VALUES
# ============================================================

import numpy as np
import pandas as pd

# Work from matrices BEFORE imputation
X_train_clean = X_train.replace(
    [np.inf, -np.inf],
    np.nan
).copy()

X_val_clean = X_val.replace(
    [np.inf, -np.inf],
    np.nan
).copy()

X_test_clean = X_test.replace(
    [np.inf, -np.inf],
    np.nan
).copy()


# ------------------------------------------------------------
# Identify clearly invalid extreme values
# float64-max-like values are not meaningful descriptors
# ------------------------------------------------------------

EXTREME_LIMIT = 1e100

for name, X in [
    ("TRAIN", X_train_clean),
    ("VALIDATION", X_val_clean),
    ("TEST", X_test_clean)
]:
    
    extreme_mask = X.abs() > EXTREME_LIMIT

    print(
        name,
        "invalid extreme values:",
        extreme_mask.sum().sum()
    )


# ------------------------------------------------------------
# Replace invalid extreme values with NaN
# ------------------------------------------------------------

X_train_clean = X_train_clean.mask(
    X_train_clean.abs() > EXTREME_LIMIT
)

X_val_clean = X_val_clean.mask(
    X_val_clean.abs() > EXTREME_LIMIT
)

X_test_clean = X_test_clean.mask(
    X_test_clean.abs() > EXTREME_LIMIT
)

TRAIN invalid extreme values: 0
VALIDATION invalid extreme values: 5
TEST invalid extreme values: 0


In [13]:
# ============================================================
# STEP 8: CLASS-WEIGHTED EXTRATREES CLASSIFIER
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. Model
# ------------------------------------------------------------

clf = ExtraTreesClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 2. Fit on TRAIN only
# ------------------------------------------------------------

clf.fit(
    X_train_proc,
    y_train
)

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------

val_pred = clf.predict(X_val_proc)
test_pred = clf.predict(X_test_proc)

val_prob = clf.predict_proba(X_val_proc)[:, 1]
test_prob = clf.predict_proba(X_test_proc)[:, 1]

# ------------------------------------------------------------
# 4. Evaluation
# ------------------------------------------------------------

def evaluate_classifier(y_true, y_pred, y_prob, name):

    print(f"\n{name}")
    print("-" * 50)

    print("Accuracy:",
          round(accuracy_score(y_true, y_pred), 4))

    print("Balanced accuracy:",
          round(balanced_accuracy_score(y_true, y_pred), 4))

    print("Precision:",
          round(precision_score(y_true, y_pred), 4))

    print("Recall:",
          round(recall_score(y_true, y_pred), 4))

    print("F1:",
          round(f1_score(y_true, y_pred), 4))

    print("MCC:",
          round(matthews_corrcoef(y_true, y_pred), 4))

    print("ROC-AUC:",
          round(roc_auc_score(y_true, y_prob), 4))

    print("PR-AUC:",
          round(average_precision_score(y_true, y_prob), 4))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["inactive", "active"]
    ))


evaluate_classifier(
    y_val,
    val_pred,
    val_prob,
    "VALIDATION"
)

evaluate_classifier(
    y_test,
    test_pred,
    test_prob,
    "TEST"
)

/Users/maryta/miniconda3/envs/mdm2_reinvent4/lib/python3.11/site-packages/sklearn/utils/_array_api.py:976: RuntimeWarning: overflow encountered in cast
  array = numpy.asarray(array, order=order, dtype=dtype)


ValueError: Input X contains infinity or a value too large for dtype('float32').

In [14]:
# ============================================================
# DIAGNOSE EXTREMELY LARGE FINITE FEATURE VALUES
# ============================================================

import numpy as np
import pandas as pd

for name, X in [
    ("TRAIN", X_train_proc),
    ("VALIDATION", X_val_proc),
    ("TEST", X_test_proc),
]:

    print(f"\n{name}")
    print("-" * 50)

    print("Maximum value:")
    print(np.nanmax(X))

    print("Minimum value:")
    print(np.nanmin(X))

    print("Maximum absolute value:")
    print(np.nanmax(np.abs(X)))

    print(
        "Values > float32 maximum:",
        np.sum(
            np.abs(X) > np.finfo(np.float32).max
        )
    )

    print(
        "Values > 1e20:",
        np.sum(np.abs(X) > 1e20)
    )

    print(
        "Values > 1e10:",
        np.sum(np.abs(X) > 1e10)
    )


TRAIN
--------------------------------------------------
Maximum value:
149413.5330233808
Minimum value:
-4848.971469172398
Maximum absolute value:
149413.5330233808
Values > float32 maximum: 0
Values > 1e20: 0
Values > 1e10: 0

VALIDATION
--------------------------------------------------
Maximum value:
1.7976931348623157e+308
Minimum value:
-1964117106.0
Maximum absolute value:
1.7976931348623157e+308
Values > float32 maximum: 5
Values > 1e20: 5
Values > 1e10: 20

TEST
--------------------------------------------------
Maximum value:
81101.56871749941
Minimum value:
-6079.690544230027
Maximum absolute value:
81101.56871749941
Values > float32 maximum: 0
Values > 1e20: 0
Values > 1e10: 0


SyntaxError: unmatched ')' (2660162661.py, line 30)

In [18]:
# ============================================================
# STEP 7B: REMOVE INVALID EXTREME PaDEL VALUES
# ============================================================

import numpy as np
import pandas as pd

# Work from matrices BEFORE imputation
X_train_clean = X_train.replace(
    [np.inf, -np.inf],
    np.nan
).copy()

X_val_clean = X_val.replace(
    [np.inf, -np.inf],
    np.nan
).copy()

X_test_clean = X_test.replace(
    [np.inf, -np.inf],
    np.nan
).copy()


# ------------------------------------------------------------
# Identify clearly invalid extreme values
# float64-max-like values are not meaningful descriptors
# ------------------------------------------------------------

EXTREME_LIMIT = 1e100

for name, X in [
    ("TRAIN", X_train_clean),
    ("VALIDATION", X_val_clean),
    ("TEST", X_test_clean)
]:
    
    extreme_mask = X.abs() > EXTREME_LIMIT

    print(
        name,
        "invalid extreme values:",
        extreme_mask.sum().sum()
    )


# ------------------------------------------------------------
# Replace invalid extreme values with NaN
# ------------------------------------------------------------

X_train_clean = X_train_clean.mask(
    X_train_clean.abs() > EXTREME_LIMIT
)

X_val_clean = X_val_clean.mask(
    X_val_clean.abs() > EXTREME_LIMIT
)

X_test_clean = X_test_clean.mask(
    X_test_clean.abs() > EXTREME_LIMIT
)

TRAIN invalid extreme values: 0
VALIDATION invalid extreme values: 5
TEST invalid extreme values: 0


In [19]:
# ============================================================
# RE-RUN TRAINING-ONLY PREPROCESSING
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold


# 1. Missing-value filter from TRAIN only
missing_fraction_train = X_train_clean.isna().mean()

keep_mask = missing_fraction_train <= 0.20
kept_columns = X_train_clean.columns[keep_mask]

X_train_f = X_train_clean[kept_columns].copy()
X_val_f   = X_val_clean[kept_columns].copy()
X_test_f  = X_test_clean[kept_columns].copy()


# 2. Median imputation — TRAIN only
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train_f)
X_val_imp   = imputer.transform(X_val_f)
X_test_imp  = imputer.transform(X_test_f)


# 3. Zero-variance filtering — TRAIN only
variance_filter = VarianceThreshold(threshold=0.0)

X_train_proc = variance_filter.fit_transform(X_train_imp)
X_val_proc   = variance_filter.transform(X_val_imp)
X_test_proc  = variance_filter.transform(X_test_imp)


# Feature names
final_feature_names = kept_columns[
    variance_filter.get_support()
].tolist()


# ------------------------------------------------------------
# Final safety checks
# ------------------------------------------------------------

print("Final features:", len(final_feature_names))

for name, X in [
    ("TRAIN", X_train_proc),
    ("VALIDATION", X_val_proc),
    ("TEST", X_test_proc)
]:

    print(f"\n{name}")
    print("NaN:", np.isnan(X).sum())
    print("Inf:", np.isinf(X).sum())
    print(
        "Values > float32 maximum:",
        np.sum(
            np.abs(X) > np.finfo(np.float32).max
        )
    )
    print(
        "Maximum absolute value:",
        np.max(np.abs(X))
    )

Final features: 3088

TRAIN
NaN: 0
Inf: 0
Values > float32 maximum: 0
Maximum absolute value: 149413.5330233808

VALIDATION
NaN: 0
Inf: 0
Values > float32 maximum: 0
Maximum absolute value: 53000010937.0

TEST
NaN: 0
Inf: 0
Values > float32 maximum: 0
Maximum absolute value: 81101.56871749941


In [20]:
# ============================================================
# STEP 8: CLASS-WEIGHTED EXTRATREES — RETRAIN
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

clf = ExtraTreesClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train_proc, y_train)

val_pred = clf.predict(X_val_proc)
test_pred = clf.predict(X_test_proc)

val_prob = clf.predict_proba(X_val_proc)[:, 1]
test_prob = clf.predict_proba(X_test_proc)[:, 1]


def evaluate_classifier(y_true, y_pred, y_prob, name):

    print(f"\n{name}")
    print("-" * 50)

    print("Accuracy:",
          round(accuracy_score(y_true, y_pred), 4))

    print("Balanced accuracy:",
          round(balanced_accuracy_score(y_true, y_pred), 4))

    print("MCC:",
          round(matthews_corrcoef(y_true, y_pred), 4))

    print("ROC-AUC:",
          round(roc_auc_score(y_true, y_prob), 4))

    print("PR-AUC (active=1):",
          round(average_precision_score(y_true, y_prob), 4))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["inactive", "active"],
            digits=4
        )
    )


evaluate_classifier(
    y_val,
    val_pred,
    val_prob,
    "VALIDATION"
)

evaluate_classifier(
    y_test,
    test_pred,
    test_prob,
    "TEST"
)


VALIDATION
--------------------------------------------------
Accuracy: 0.9683
Balanced accuracy: 0.7
MCC: 0.4977
ROC-AUC: 0.9241
PR-AUC (active=1): 0.9941

Confusion matrix:
[[  9  13]
 [  5 540]]

Classification report:
              precision    recall  f1-score   support

    inactive     0.6429    0.4091    0.5000        22
      active     0.9765    0.9908    0.9836       545

    accuracy                         0.9683       567
   macro avg     0.8097    0.7000    0.7418       567
weighted avg     0.9635    0.9683    0.9648       567


TEST
--------------------------------------------------
Accuracy: 0.965
Balanced accuracy: 0.7169
MCC: 0.6046
ROC-AUC: 0.9732
PR-AUC (active=1): 0.9976

Confusion matrix:
[[ 14  18]
 [  2 537]]

Classification report:
              precision    recall  f1-score   support

    inactive     0.8750    0.4375    0.5833        32
      active     0.9676    0.9963    0.9817       539

    accuracy                         0.9650       571
   macro avg 

In [21]:
print("TEST confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nTEST classification report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["inactive", "active"],
        digits=4
    )
)

print("TEST balanced accuracy:",
      balanced_accuracy_score(y_test, test_pred))

print("TEST MCC:",
      matthews_corrcoef(y_test, test_pred))

print("TEST ROC-AUC:",
      roc_auc_score(y_test, test_prob))

TEST confusion matrix:
[[ 14  18]
 [  2 537]]

TEST classification report:
              precision    recall  f1-score   support

    inactive     0.8750    0.4375    0.5833        32
      active     0.9676    0.9963    0.9817       539

    accuracy                         0.9650       571
   macro avg     0.9213    0.7169    0.7825       571
weighted avg     0.9624    0.9650    0.9594       571

TEST balanced accuracy: 0.7168947124304268
TEST MCC: 0.6045633966273044
TEST ROC-AUC: 0.9732432745825603


In [22]:
# ============================================================
# STEP 9: OPTIMIZE CLASSIFICATION THRESHOLD ON VALIDATION ONLY
# Criterion: maximize MCC
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

thresholds = np.arange(
    0.05,
    0.951,
    0.01
)

results = []

for threshold in thresholds:

    val_pred_t = (
        val_prob >= threshold
    ).astype(int)

    results.append({
        "threshold": threshold,
        "MCC": matthews_corrcoef(
            y_val,
            val_pred_t
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                val_pred_t
            )
    })

threshold_results = pd.DataFrame(results)

best_row = (
    threshold_results
    .sort_values(
        "MCC",
        ascending=False
    )
    .iloc[0]
)

best_threshold = best_row["threshold"]

print(
    "Best validation threshold:",
    round(best_threshold, 2)
)

print(
    "Validation MCC:",
    round(best_row["MCC"], 4)
)

print(
    "Validation balanced accuracy:",
    round(
        best_row["balanced_accuracy"],
        4
    )
)


# ------------------------------------------------------------
# Validation performance at selected threshold
# ------------------------------------------------------------

val_pred_opt = (
    val_prob >= best_threshold
).astype(int)

print("\nVALIDATION confusion matrix:")
print(
    confusion_matrix(
        y_val,
        val_pred_opt
    )
)

print("\nVALIDATION report:")
print(
    classification_report(
        y_val,
        val_pred_opt,
        target_names=[
            "inactive",
            "active"
        ],
        digits=4
    )
)

Best validation threshold: 0.65
Validation MCC: 0.6708
Validation balanced accuracy: 0.8763

VALIDATION confusion matrix:
[[ 17   5]
 [ 11 534]]

VALIDATION report:
              precision    recall  f1-score   support

    inactive     0.6071    0.7727    0.6800        22
      active     0.9907    0.9798    0.9852       545

    accuracy                         0.9718       567
   macro avg     0.7989    0.8763    0.8326       567
weighted avg     0.9758    0.9718    0.9734       567



In [23]:
# ============================================================
# STEP 9: OPTIMIZE CLASSIFICATION THRESHOLD ON VALIDATION ONLY
# Criterion: maximize MCC
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

thresholds = np.arange(
    0.05,
    0.951,
    0.01
)

results = []

for threshold in thresholds:

    val_pred_t = (
        val_prob >= threshold
    ).astype(int)

    results.append({
        "threshold": threshold,
        "MCC": matthews_corrcoef(
            y_val,
            val_pred_t
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                val_pred_t
            )
    })

threshold_results = pd.DataFrame(results)

best_row = (
    threshold_results
    .sort_values(
        "MCC",
        ascending=False
    )
    .iloc[0]
)

best_threshold = best_row["threshold"]

print(
    "Best validation threshold:",
    round(best_threshold, 2)
)

print(
    "Validation MCC:",
    round(best_row["MCC"], 4)
)

print(
    "Validation balanced accuracy:",
    round(
        best_row["balanced_accuracy"],
        4
    )
)


# ------------------------------------------------------------
# Validation performance at selected threshold
# ------------------------------------------------------------

val_pred_opt = (
    val_prob >= best_threshold
).astype(int)

print("\nVALIDATION confusion matrix:")
print(
    confusion_matrix(
        y_val,
        val_pred_opt
    )
)

print("\nVALIDATION report:")
print(
    classification_report(
        y_val,
        val_pred_opt,
        target_names=[
            "inactive",
            "active"
        ],
        digits=4
    )
)

Best validation threshold: 0.65
Validation MCC: 0.6708
Validation balanced accuracy: 0.8763

VALIDATION confusion matrix:
[[ 17   5]
 [ 11 534]]

VALIDATION report:
              precision    recall  f1-score   support

    inactive     0.6071    0.7727    0.6800        22
      active     0.9907    0.9798    0.9852       545

    accuracy                         0.9718       567
   macro avg     0.7989    0.8763    0.8326       567
weighted avg     0.9758    0.9718    0.9734       567



In [24]:
# ============================================================
# STEP 10: APPLY VALIDATION-SELECTED THRESHOLD TO TEST
# ============================================================

best_threshold = 0.65

test_pred_opt = (
    test_prob >= best_threshold
).astype(int)

print("TEST confusion matrix:")
print(
    confusion_matrix(
        y_test,
        test_pred_opt
    )
)

print("\nTEST classification report:")
print(
    classification_report(
        y_test,
        test_pred_opt,
        target_names=[
            "inactive",
            "active"
        ],
        digits=4
    )
)

print(
    "TEST balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            test_pred_opt
        ),
        4
    )
)

print(
    "TEST MCC:",
    round(
        matthews_corrcoef(
            y_test,
            test_pred_opt
        ),
        4
    )
)

print(
    "TEST ROC-AUC:",
    round(
        roc_auc_score(
            y_test,
            test_prob
        ),
        4
    )
)

TEST confusion matrix:
[[ 22  10]
 [  7 532]]

TEST classification report:
              precision    recall  f1-score   support

    inactive     0.7586    0.6875    0.7213        32
      active     0.9815    0.9870    0.9843       539

    accuracy                         0.9702       571
   macro avg     0.8701    0.8373    0.8528       571
weighted avg     0.9691    0.9702    0.9695       571

TEST balanced accuracy: 0.8373
TEST MCC: 0.7066
TEST ROC-AUC: 0.9732


In [25]:
# ============================================================
# STEP 11: BENCHMARK CLASS-WEIGHTED CLASSIFIERS
# Validation set only
# ============================================================

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)


models = {

    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}


results = []


for model_name, model in models.items():

    print(f"Training: {model_name}")

    model.fit(
        X_train_proc,
        y_train
    )

    pred = model.predict(
        X_val_proc
    )

    prob = model.predict_proba(
        X_val_proc
    )[:, 1]


    # inactive = 0, therefore evaluate it explicitly
    inactive_precision = precision_score(
        y_val,
        pred,
        pos_label=0,
        zero_division=0
    )

    inactive_recall = recall_score(
        y_val,
        pred,
        pos_label=0,
        zero_division=0
    )

    inactive_f1 = f1_score(
        y_val,
        pred,
        pos_label=0,
        zero_division=0
    )


    results.append({

        "Model": model_name,

        "Accuracy":
            accuracy_score(
                y_val,
                pred
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_val,
                pred
            ),

        "MCC":
            matthews_corrcoef(
                y_val,
                pred
            ),

        "ROC-AUC":
            roc_auc_score(
                y_val,
                prob
            ),

        "Inactive Precision":
            inactive_precision,

        "Inactive Recall":
            inactive_recall,

        "Inactive F1":
            inactive_f1
    })


benchmark_results = pd.DataFrame(
    results
)

benchmark_results = (
    benchmark_results
    .sort_values(
        "MCC",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nVALIDATION BENCHMARK")
display(
    benchmark_results.round(4)
)

Training: Logistic Regression


/Users/maryta/miniconda3/envs/mdm2_reinvent4/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training: Random Forest
Training: ExtraTrees

VALIDATION BENCHMARK


,Model,Accuracy,Balanced Accuracy,MCC,ROC-AUC,Inactive Precision,Inactive Recall,Inactive F1
0,Random Forest,0.9735,0.7681,0.6073,0.9715,0.7059,0.5455,0.6154
1,Logistic Regression,0.9524,0.8226,0.5177,0.9648,0.4286,0.6818,0.5263
2,ExtraTrees,0.9683,0.7000,0.4977,0.9241,0.6429,0.4091,0.5000


In [26]:
# ============================================================
# STEP 12: OPTIMIZE VALIDATION THRESHOLD FOR EACH MODEL
# Criterion: maximum MCC
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

thresholds = np.arange(0.05, 0.951, 0.01)

optimized_results = []

for model_name, model in models.items():

    # Probability of ACTIVE class = 1
    val_prob_model = model.predict_proba(
        X_val_proc
    )[:, 1]

    best_result = None

    for threshold in thresholds:

        pred = (
            val_prob_model >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_val,
            pred
        )

        if (
            best_result is None
            or mcc > best_result["MCC"]
        ):

            best_result = {
                "Model": model_name,
                "Threshold": threshold,
                "MCC": mcc,

                "Balanced Accuracy":
                    balanced_accuracy_score(
                        y_val,
                        pred
                    ),

                "Inactive Precision":
                    precision_score(
                        y_val,
                        pred,
                        pos_label=0,
                        zero_division=0
                    ),

                "Inactive Recall":
                    recall_score(
                        y_val,
                        pred,
                        pos_label=0,
                        zero_division=0
                    ),

                "Inactive F1":
                    f1_score(
                        y_val,
                        pred,
                        pos_label=0,
                        zero_division=0
                    ),

                "ROC-AUC":
                    roc_auc_score(
                        y_val,
                        val_prob_model
                    )
            }

    optimized_results.append(
        best_result
    )


optimized_benchmark = (
    pd.DataFrame(
        optimized_results
    )
    .sort_values(
        "MCC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    optimized_benchmark.round(4)
)

,Model,Threshold,MCC,Balanced Accuracy,Inactive Precision,Inactive Recall,Inactive F1,ROC-AUC
0,ExtraTrees,0.65,0.6708,0.8763,0.6071,0.7727,0.6800,0.9241
1,Random Forest,0.79,0.6401,0.9144,0.5000,0.8636,0.6333,0.9715
2,Logistic Regression,0.24,0.5352,0.8244,0.4545,0.6818,0.5455,0.9648


In [27]:
# ============================================================
# STEP 12: OPTIMIZE VALIDATION THRESHOLD FOR EACH MODEL
# Criterion: maximum MCC
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

thresholds = np.arange(0.05, 0.951, 0.01)

optimized_results = []

for model_name, model in models.items():

    # Probability of ACTIVE class = 1
    val_prob_model = model.predict_proba(
        X_val_proc
    )[:, 1]

    best_result = None

    for threshold in thresholds:

        pred = (
            val_prob_model >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_val,
            pred
        )

        if (
            best_result is None
            or mcc > best_result["MCC"]
        ):

            best_result = {
                "Model": model_name,
                "Threshold": threshold,
                "MCC": mcc,

                "Balanced Accuracy":
                    balanced_accuracy_score(
                        y_val,
                        pred
                    ),

                "Inactive Precision":
                    precision_score(
                        y_val,
                        pred,
                        pos_label=0,
                        zero_division=0
                    ),

                "Inactive Recall":
                    recall_score(
                        y_val,
                        pred,
                        pos_label=0,
                        zero_division=0
                    ),

                "Inactive F1":
                    f1_score(
                        y_val,
                        pred,
                        pos_label=0,
                        zero_division=0
                    ),

                "ROC-AUC":
                    roc_auc_score(
                        y_val,
                        val_prob_model
                    )
            }

    optimized_results.append(
        best_result
    )


optimized_benchmark = (
    pd.DataFrame(
        optimized_results
    )
    .sort_values(
        "MCC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    optimized_benchmark.round(4)
)

,Model,Threshold,MCC,Balanced Accuracy,Inactive Precision,Inactive Recall,Inactive F1,ROC-AUC
0,ExtraTrees,0.65,0.6708,0.8763,0.6071,0.7727,0.6800,0.9241
1,Random Forest,0.79,0.6401,0.9144,0.5000,0.8636,0.6333,0.9715
2,Logistic Regression,0.24,0.5352,0.8244,0.4545,0.6818,0.5455,0.9648


In [28]:
# ============================================================
# STEP 13: LAZYPREDICT CLASSIFICATION BENCHMARK
# TRAIN -> VALIDATION ONLY
# ============================================================

from lazypredict.Supervised import LazyClassifier

lazy_clf = LazyClassifier(
    verbose=0,
    ignore_warnings=True,
    custom_metric=None,
    predictions=False,
    random_state=42
)

lazy_models, _ = lazy_clf.fit(
    X_train_proc,
    X_val_proc,
    y_train,
    y_val
)

print("LazyPredict validation benchmark:")

display(
    lazy_models.sort_values(
        by="Balanced Accuracy",
        ascending=False
    )
)


High-dimensional dataset (3088 features). Some models may be slow or fail. Consider dimensionality reduction.


LazyPredict validation benchmark:


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
BernoulliNB,0.823633,0.886447,0.923603,0.875780,0.965839,0.823633,0.232864
PassiveAggressiveClassifier,0.955908,0.846205,0.933653,0.960670,0.968096,0.955908,0.533280
ExtraTreeClassifier,0.964727,0.763553,0.826814,0.964727,0.964727,0.964727,0.159442
DecisionTreeClassifier,0.943563,0.752544,0.794579,0.949295,0.956860,0.943563,2.128567
RidgeClassifier,0.934744,0.747957,0.852502,0.943320,0.954933,0.934744,0.620747
RidgeClassifierCV,0.955908,0.737156,0.881485,0.957255,0.958764,0.955908,2.398853
XGBClassifier,0.970018,0.722686,0.979399,0.967277,0.966171,0.970018,2.972350
BaggingClassifier,0.966490,0.720851,0.876856,0.964419,0.963052,0.966490,12.436787
LinearSVC,0.941799,0.708007,0.797706,0.946498,0.952196,0.941799,9.866594


In [30]:
# ============================================================
# STEP 14: EXTRATREES FEATURE-IMPORTANCE REDUCTION
# Train only -> Validation comparison
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Fit ExtraTrees on ALL processed training features
# ------------------------------------------------------------

feature_selector_model = ExtraTreesClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

feature_selector_model.fit(
    X_train_proc,
    y_train
)

# ------------------------------------------------------------
# 2. Rank features by importance
# ------------------------------------------------------------

feature_importance_df = pd.DataFrame({
    "feature": final_feature_names,
    "importance": feature_selector_model.feature_importances_
})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance_df["cumulative_importance"] = (
    feature_importance_df["importance"]
    .cumsum()
)

print("Total features:", len(feature_importance_df))

display(
    feature_importance_df.head(20)
)

Total features: 3088


,feature,importance,cumulative_importance
0,Morgan_r2_bit_1671,0.013519,0.013519
1,Morgan_r2_bit_1855,0.010894,0.024413
2,Morgan_r2_bit_0352,0.010831,0.035244
3,Morgan_r2_bit_0378,0.010145,0.045389
4,Morgan_r2_bit_1978,0.007346,0.052735
5,SRW5,0.007320,0.060056
6,Morgan_r2_bit_0802,0.006749,0.066805
7,Morgan_r2_bit_0323,0.006291,0.073096
8,maxaaN,0.006205,0.079301
9,Morgan_r2_bit_0690,0.006024,0.085325


In [31]:
# ============================================================
# STEP 15: TEST DIFFERENT CUMULATIVE IMPORTANCE LEVELS
# Validation only
# ============================================================

threshold_levels = [
    0.95,
    0.975,
    0.99,
    1.00
]

threshold_grid = np.arange(
    0.05,
    0.951,
    0.01
)

reduction_results = []

selected_feature_sets = {}

for importance_level in threshold_levels:

    # --------------------------------------------------------
    # Determine number of features needed
    # --------------------------------------------------------

    if importance_level < 1.0:

        n_features = (
            np.searchsorted(
                feature_importance_df[
                    "cumulative_importance"
                ].values,
                importance_level
            )
            + 1
        )

    else:
        n_features = len(
            feature_importance_df
        )

    # Get selected feature names
    selected_names = (
        feature_importance_df
        .iloc[:n_features]["feature"]
        .tolist()
    )

    selected_feature_sets[
        importance_level
    ] = selected_names

    # --------------------------------------------------------
    # Convert names to column indices
    # --------------------------------------------------------

    feature_index_map = {
        name: i
        for i, name in enumerate(
            final_feature_names
        )
    }

    selected_indices = [
        feature_index_map[name]
        for name in selected_names
    ]

    X_train_selected = (
        X_train_proc[:, selected_indices]
    )

    X_val_selected = (
        X_val_proc[:, selected_indices]
    )

    # --------------------------------------------------------
    # Train a fresh ExtraTrees model
    # --------------------------------------------------------

    model = ExtraTreesClassifier(
        n_estimators=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_selected,
        y_train
    )

    val_prob_selected = (
        model.predict_proba(
            X_val_selected
        )[:, 1]
    )

    # --------------------------------------------------------
    # Optimize classification threshold
    # using VALIDATION MCC only
    # --------------------------------------------------------

    best_mcc = -np.inf
    best_threshold = None
    best_pred = None

    for decision_threshold in threshold_grid:

        pred = (
            val_prob_selected
            >= decision_threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_val,
            pred
        )

        if mcc > best_mcc:

            best_mcc = mcc
            best_threshold = (
                decision_threshold
            )
            best_pred = pred

    # --------------------------------------------------------
    # Store validation results
    # --------------------------------------------------------

    reduction_results.append({

        "Importance retained":
            importance_level,

        "Number of features":
            n_features,

        "Reduction %":
            100 * (
                1 -
                n_features /
                len(final_feature_names)
            ),

        "Best threshold":
            best_threshold,

        "MCC":
            best_mcc,

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_val,
                best_pred
            ),

        "Inactive Precision":
            precision_score(
                y_val,
                best_pred,
                pos_label=0,
                zero_division=0
            ),

        "Inactive Recall":
            recall_score(
                y_val,
                best_pred,
                pos_label=0,
                zero_division=0
            ),

        "Inactive F1":
            f1_score(
                y_val,
                best_pred,
                pos_label=0,
                zero_division=0
            ),

        "ROC-AUC":
            roc_auc_score(
                y_val,
                val_prob_selected
            )
    })


reduction_results_df = pd.DataFrame(
    reduction_results
)

print(
    "\nFEATURE REDUCTION — VALIDATION RESULTS"
)

display(
    reduction_results_df.round(4)
)


FEATURE REDUCTION — VALIDATION RESULTS


,Importance retained,Number of features,Reduction %,Best threshold,MCC,Balanced Accuracy,Inactive Precision,Inactive Recall,Inactive F1,ROC-AUC
0,0.950,1356,56.0881,0.68,0.6708,0.8763,0.6071,0.7727,0.6800,0.9287
1,0.975,1542,50.0648,0.66,0.6708,0.8763,0.6071,0.7727,0.6800,0.9154
2,0.990,1747,43.4262,0.66,0.6708,0.8763,0.6071,0.7727,0.6800,0.9273
3,1.000,3088,0.0000,0.67,0.6581,0.8754,0.5862,0.7727,0.6667,0.9153


In [33]:
# ============================================================
# STEP 16: HYPERPARAMETER TUNING
# ExtraTreesClassifier on 95% importance feature set
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, matthews_corrcoef


# ------------------------------------------------------------
# 1. Get the selected 95% feature set
# ------------------------------------------------------------

selected_names_95 = selected_feature_sets[0.95]

feature_index_map = {
    name: i
    for i, name in enumerate(final_feature_names)
}

selected_indices_95 = [
    feature_index_map[name]
    for name in selected_names_95
]

X_train_95 = X_train_proc[:, selected_indices_95]
X_val_95   = X_val_proc[:, selected_indices_95]
X_test_95  = X_test_proc[:, selected_indices_95]

print("Train shape:", X_train_95.shape)
print("Validation shape:", X_val_95.shape)
print("Test shape:", X_test_95.shape)


# ------------------------------------------------------------
# 2. ExtraTrees model
# ------------------------------------------------------------

base_model = ExtraTreesClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# 3. Hyperparameter grid
# ------------------------------------------------------------

param_grid = {

    "n_estimators": [
        500,
        1000
    ],

    "max_features": [
        "sqrt",
        0.2,
        0.5
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ],

    "max_depth": [
        None,
        20,
        40
    ]
}


# ------------------------------------------------------------
# 4. MCC scoring
# ------------------------------------------------------------

mcc_scorer = make_scorer(
    matthews_corrcoef
)


# ------------------------------------------------------------
# 5. Stratified cross-validation
# Training data ONLY
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ------------------------------------------------------------
# 6. Grid search
# ------------------------------------------------------------

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring=mcc_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)


grid_search.fit(
    X_train_95,
    y_train
)


# ------------------------------------------------------------
# 7. Best result
# ------------------------------------------------------------

print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation MCC:")
print(
    round(
        grid_search.best_score_,
        4
    )
)


best_et_model = (
    grid_search.best_estimator_
)

Train shape: (2619, 1356)
Validation shape: (567, 1356)
Test shape: (571, 1356)
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time=   3.1s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time=   3.1s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time=   3.4s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time=   3.5s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time=   3.9s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=1000; total time=   6.0s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=500; total time=   2.8s
[C